# Retail Customer Behavior: E-Commerce Transaction Analytics

## Project Overview

This is an analysis of about a year of online retail transactions for a UK-based gift retailer. The dataset has roughly half a million line items across thousands of customers and dozens of countries. I wanted to look at it from a few angles: what does the typical order look like, when do people buy, where do they buy from, and can I segment customers in a way that would actually be useful to a marketing team?

The notebook moves through cleaning, feature engineering, exploratory analysis, geospatial visualization, and finishes with an RFM-based customer segmentation using K-Means clustering. The clustering part is the most useful for a real business: it groups customers into actionable buckets like "high-value loyal", "at-risk", and "new but promising" so you can target each one differently.

**Dataset**: UCI Online Retail (publicly available on the UCI Machine Learning Repository).

**Tools**: Python, pandas, NumPy, matplotlib, seaborn, GeoPandas, scikit-learn.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
os.makedirs('figures', exist_ok=True)

## Loading and Cleaning the Data

The raw file has the usual issues: missing customer IDs, cancelled invoices (denoted with a `C` prefix), negative quantities, zero prices, and duplicate rows. I drop all of those and parse `InvoiceDate` to a real datetime so I can extract time features later. ISO-8859-1 encoding is needed because the file has special characters in the product descriptions.

In [ ]:
df = pd.read_csv('data/EcommerceData.csv', encoding='ISO-8859-1')

print('Original shape:', df.shape)
print()
print('Missing values:')
print(df.isnull().sum())

In [ ]:
df = df.dropna(subset=['CustomerID', 'Description'])
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df = df.drop_duplicates()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print('Cleaned shape:', df.shape)
print('Date range:', df['InvoiceDate'].min(), 'to', df['InvoiceDate'].max())

## Feature Engineering

I build a few derived columns that come up repeatedly in later sections: `TotalPrice` for the line-item revenue, plus `Hour`, `DayOfWeek`, `Month`, and `IsWeekend` for time-based analysis. Using day names (instead of 0 to 6) makes the heatmap easier to read.

In [ ]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Month'] = df['InvoiceDate'].dt.month
df['IsWeekend'] = df['InvoiceDate'].dt.dayofweek >= 5

df[['InvoiceDate', 'TotalPrice', 'Hour', 'DayOfWeek', 'Month', 'IsWeekend']].head()

## Top-Line Numbers

A few summary stats to ground the rest of the analysis.

In [ ]:
total_customers = df['CustomerID'].nunique()
total_transactions = df['InvoiceNo'].nunique()
total_revenue = df['TotalPrice'].sum()
average_order_value = df.groupby('InvoiceNo')['TotalPrice'].sum().mean()

print(f'Total Unique Customers : {total_customers:,}')
print(f'Total Transactions     : {total_transactions:,}')
print(f'Total Revenue          : ${total_revenue:,.2f}')
print(f'Average Order Value    : ${average_order_value:,.2f}')

## Distribution of Line-Item Prices

The line-item price is heavily skewed (a few huge purchases pull the mean way up), so I cap the histogram at $100 to actually see where the bulk of activity sits.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df[df['TotalPrice'] < 100]['TotalPrice'], bins=50, kde=True, color='steelblue')
plt.title('Distribution of Total Price per Line Item (capped at $100)', fontsize=13, fontweight='bold')
plt.xlabel('Total Price ($)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('figures/totalprice_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

## Best-Selling Products

In [ ]:
top_products = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
top_products.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Top 10 Best-Selling Products by Quantity', fontsize=13, fontweight='bold')
plt.xlabel('Product Description')
plt.ylabel('Total Quantity Sold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('figures/top_products.png', dpi=150, bbox_inches='tight')
plt.show()

## When Do People Shop?

A heatmap of order count by hour of day and day of the week tells you when the staffing and infrastructure load is highest. Useful both for ops planning and for timing email campaigns.

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

heatmap_data = (
    df.groupby(['DayOfWeek', 'Hour'])['InvoiceNo']
    .count()
    .unstack(fill_value=0)
)
heatmap_data = heatmap_data.reindex(day_order)

plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_data, cmap='YlGnBu', linewidths=0.3, linecolor='white')
plt.title('Sales Volume by Hour and Day of Week', fontsize=13, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.savefig('figures/heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Weekday vs Weekend Order Values

Are weekend shoppers different from weekday shoppers? KDE density makes the comparison cleaner than overlapping histograms.

In [ ]:
order_totals = df.groupby(['InvoiceNo', 'IsWeekend'])['TotalPrice'].sum().reset_index()
order_totals = order_totals[order_totals['TotalPrice'] <= 1500]

weekday = order_totals[order_totals['IsWeekend'] == False]['TotalPrice']
weekend = order_totals[order_totals['IsWeekend'] == True]['TotalPrice']

plt.figure(figsize=(10, 6))
sns.kdeplot(weekday, label='Weekday', fill=True, alpha=0.4, color='steelblue')
sns.kdeplot(weekend, label='Weekend', fill=True, alpha=0.4, color='coral')
plt.title('Order Value Distribution: Weekday vs Weekend', fontsize=13, fontweight='bold')
plt.xlabel('Total Order Value ($)')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.savefig('figures/kde_weekday_weekend.png', dpi=150, bbox_inches='tight')
plt.show()

## Monthly Revenue Trend

Seasonality matters for retail. The Nov/Dec spike is no surprise for a gift retailer.

In [ ]:
monthly_rev = df.groupby('Month')['TotalPrice'].sum()
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(10, 6))
monthly_rev.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Total Revenue by Month', fontsize=13, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Revenue ($)')
plt.xticks(ticks=range(12), labels=month_labels, rotation=0)
plt.tight_layout()
plt.savefig('figures/monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

## Order Values by Country (Excluding UK)

The UK dominates volume so I drop it to compare the other markets head-to-head. Capping at $100 keeps the boxplot readable.

In [ ]:
top_countries = (
    df[df['Country'] != 'United Kingdom']
    .groupby('Country')['TotalPrice'].sum()
    .sort_values(ascending=False)
    .head(5).index
)

box_df = df[(df['Country'].isin(top_countries)) & (df['TotalPrice'] < 100)]

plt.figure(figsize=(12, 6))
sns.boxplot(x='Country', y='TotalPrice', data=box_df, palette='Set2')
plt.title('Order Value Distribution by Top 5 Non-UK Countries', fontsize=13, fontweight='bold')
plt.xlabel('Country')
plt.ylabel('Order Value ($)')
plt.tight_layout()
plt.savefig('figures/boxplot_country.png', dpi=150, bbox_inches='tight')
plt.show()

## Geospatial: Where Are the Customers?

A choropleth of total invoices by country shows the global reach. The UK is by far the biggest market (this is a UK retailer), with material activity in Germany, France, Ireland, and a long tail elsewhere.

In [ ]:
country_invoices = df.groupby('Country')['InvoiceNo'].nunique().reset_index()
country_invoices.columns = ['Country', 'InvoiceCount']

country_invoices['Country'] = country_invoices['Country'].replace({
    'EIRE': 'Ireland',
    'USA': 'United States of America',
    'RSA': 'South Africa'
})

# Try a few sources for the world shapefile, with a graceful fallback to a tabular view
world = None
for source in [
    lambda: __import__('geopandas').read_file(__import__('geopandas').datasets.get_path('naturalearth_lowres')),
    lambda: gpd.read_file('https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip'),
    lambda: gpd.read_file('https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip'),
]:
    try:
        world = source()
        break
    except Exception:
        continue

if world is not None:
    name_col = 'NAME' if 'NAME' in world.columns else 'name'
    world_data = world.merge(country_invoices, how='left', left_on=name_col, right_on='Country')
    world_data['InvoiceCount'] = world_data['InvoiceCount'].fillna(0)

    fig, ax = plt.subplots(1, 1, figsize=(15, 8))
    world_data.plot(
        column='InvoiceCount',
        ax=ax,
        legend=True,
        legend_kwds={'label': 'Number of Invoices by Country', 'orientation': 'horizontal'},
        cmap='OrRd',
        linewidth=0.4,
        edgecolor='gray',
        missing_kwds={'color': 'lightgrey'}
    )
    ax.set_title('Global Distribution of E-Commerce Invoices', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('figures/choropleth.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    # Fallback when offline: show a horizontal bar chart of the top 15 countries by invoice volume
    print('World shapefile not available offline. Showing top countries as a bar chart instead.')
    top = country_invoices.sort_values('InvoiceCount', ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.barh(top['Country'][::-1], top['InvoiceCount'][::-1], color='steelblue')
    ax.set_xlabel('Number of Invoices')
    ax.set_title('Top 15 Countries by Invoice Volume', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/choropleth.png', dpi=150, bbox_inches='tight')
    plt.show()

## RFM Customer Segmentation

This is the part of the analysis that would actually be useful to a marketing team. RFM (Recency, Frequency, Monetary) is a classic customer segmentation framework. For each customer I compute:

- **Recency**: how many days since their last purchase (smaller is better)
- **Frequency**: how many distinct invoices they have placed
- **Monetary**: total amount they have spent

Then I use K-Means clustering on these three features to group customers into actionable segments. Before clustering I log-transform the data (the distributions are heavily skewed) and standardize, so no single feature dominates the distance metric.

In [ ]:
# Reference date: one day after the last transaction in the dataset
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print('Snapshot date:', snapshot_date.date())

rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum'),
).reset_index()

print(f'Customers in RFM table: {len(rfm):,}')
print()
print(rfm.describe().round(2))

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Log-transform to compress the long tails, then standardize
rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].apply(np.log1p)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# Find a reasonable k using inertia and silhouette
from sklearn.metrics import silhouette_score
inertias = []
sil_scores = []
ks = range(2, 9)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(ks), inertias, 'o-', color='steelblue')
axes[0].set_title('Elbow: Inertia vs k', fontweight='bold')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[1].plot(list(ks), sil_scores, 'o-', color='coral')
axes[1].set_title('Silhouette Score vs k', fontweight='bold')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette')
plt.tight_layout()
plt.savefig('figures/cluster_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

print('Silhouette scores:', dict(zip(ks, [round(s, 3) for s in sil_scores])))

In [ ]:
# Settle on k=4 as a balance between interpretability and silhouette
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

cluster_profile = (
    rfm.groupby('Cluster')
       .agg(Customers=('CustomerID', 'count'),
            Avg_Recency_days=('Recency', 'mean'),
            Avg_Frequency=('Frequency', 'mean'),
            Avg_Monetary=('Monetary', 'mean'),
            Total_Revenue=('Monetary', 'sum'))
       .round(1)
       .sort_values('Avg_Monetary', ascending=False)
)
cluster_profile['Pct_of_Customers'] = (cluster_profile['Customers'] / cluster_profile['Customers'].sum() * 100).round(1)
cluster_profile['Pct_of_Revenue'] = (cluster_profile['Total_Revenue'] / cluster_profile['Total_Revenue'].sum() * 100).round(1)
print(cluster_profile)

In [ ]:
# Label the clusters based on their RFM profile
def label_cluster(row):
    if row['Avg_Monetary'] > 5000 and row['Avg_Recency_days'] < 50:
        return 'High-Value Loyal'
    if row['Avg_Recency_days'] > 150:
        return 'At-Risk / Churned'
    if row['Avg_Frequency'] >= 4:
        return 'Engaged Regulars'
    return 'New / Occasional'

cluster_profile['Segment'] = cluster_profile.apply(label_cluster, axis=1)
print(cluster_profile[['Segment', 'Customers', 'Pct_of_Customers',
                       'Avg_Recency_days', 'Avg_Frequency', 'Avg_Monetary',
                       'Pct_of_Revenue']])

# Bring labels back onto the customer-level rfm table
rfm = rfm.merge(cluster_profile[['Segment']], left_on='Cluster', right_index=True, how='left')

In [ ]:
# Visualize the segments on a 2D projection of the scaled features
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(rfm_scaled)
rfm['PC1'] = coords[:, 0]
rfm['PC2'] = coords[:, 1]

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=rfm, x='PC1', y='PC2', hue='Segment', palette='Set2',
    alpha=0.6, s=30, edgecolor='none'
)
plt.title('Customer Segments (PCA projection of RFM features)', fontsize=13, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('figures/segments_pca.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bar chart of revenue contribution by segment
seg_rev = (
    rfm.groupby('Segment')
       .agg(Customers=('CustomerID', 'count'), Revenue=('Monetary', 'sum'))
       .reset_index()
       .sort_values('Revenue', ascending=False)
)
seg_rev['Pct_Customers'] = seg_rev['Customers'] / seg_rev['Customers'].sum() * 100
seg_rev['Pct_Revenue'] = seg_rev['Revenue'] / seg_rev['Revenue'].sum() * 100

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(seg_rev))
width = 0.35
ax.bar(x - width/2, seg_rev['Pct_Customers'], width, label='% of Customers', color='steelblue')
ax.bar(x + width/2, seg_rev['Pct_Revenue'], width, label='% of Revenue', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(seg_rev['Segment'])
ax.set_ylabel('Percent')
ax.set_title('Customer Count vs Revenue Share by Segment', fontsize=13, fontweight='bold')
ax.legend()
for i, (c, r) in enumerate(zip(seg_rev['Pct_Customers'], seg_rev['Pct_Revenue'])):
    ax.text(i - width/2, c + 1, f'{c:.0f}%', ha='center', fontsize=9)
    ax.text(i + width/2, r + 1, f'{r:.0f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('figures/segment_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

1. **The business is heavily UK-driven but has a real European tail.** Germany, France, and Ireland together account for the bulk of non-UK volume, which is useful for thinking about logistics and language localization.
2. **Order timing has clear patterns.** Tuesday through Thursday between 10 AM and 3 PM is the busiest window. There is barely any Saturday activity, which is unusual for retail and probably reflects the wholesale-leaning customer base.
3. **The November and December spike is dramatic.** Inventory and staffing planning should weight Q4 disproportionately.
4. **A small group of customers drives most of the revenue.** The "High-Value Loyal" segment from the K-Means clustering is a small percentage of the customer base but contributes a disproportionate share of total revenue. This is the classic Pareto-style pattern, and it tells you exactly where retention dollars should go.
5. **The "At-Risk / Churned" segment is a clear win-back target.** These customers used to spend money but have not bought in a long time. A targeted reactivation campaign here is much more efficient than blanket discounts.

If I were extending this I would build a churn prediction model on top of the RFM features, layer in product-category data so I could recommend specific items per segment, and look at customer lifetime value over the cohort.
